In [10]:
import sys

!{sys.executable} -m pip install -U opencv-python ultralytics

Defaulting to user installation because normal site-packages is not writeable


In [1]:
import cv2
import warnings
import os
from ultralytics import YOLO

# Suppress PyTorch and system warning notifications
warnings.filterwarnings("ignore", category=UserWarning)

def visualize_red_car_tracking(video_path):
    # Enforce stable CPU operations
    model = YOLO("yolov8n.pt")
    model.to('cpu')
   
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video at {video_path}")
        return

    frame_count = 0
    red_car_frame_count = 0

    # Red wraps around the HSV spectrum, requiring two separate boundary ranges
    # Lower Red range (0 to 10 Hue)
    lower_red1 = (0, 50, 50)
    upper_red1 = (10, 255, 255)
   
    # Upper Red range (170 to 180 Hue)
    lower_red2 = (170, 50, 50)
    upper_red2 = (180, 255, 255)

    print("Opening video playback tracker window. Press 'q' to quit.")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
           
        frame_count += 1
        red_car_detected_in_frame = False

        # Run model inference without redundant terminal printing
        results = model(frame, verbose=False)
       
        for result in results:
            boxes = result.boxes
            for box in boxes:
                # Class 2 corresponds to 'car' in the COCO dataset mapping
                if int(box.cls) == 2:  
                    # Extract 1D array coordinates correctly out of the nested list
                    coords = box.xyxy.tolist()[0]
                    x1, y1, x2, y2 = map(int, coords)
                   
                    conf = float(box.conf[0])
                    car_crop = frame[y1:y2, x1:x2]
                   
                    if car_crop.size == 0:
                        continue
                       
                    # Extract dimensions to calculate total surface area pixels
                    height, width, _ = car_crop.shape
                    total_pixels = height * width
                   
                    # Convert car crop to HSV color profile
                    hsv_car = cv2.cvtColor(car_crop, cv2.COLOR_BGR2HSV)
                   
                    # Create two distinct masks for both red hue spectrum boundaries
                    mask1 = cv2.inRange(hsv_car, lower_red1, upper_red1)
                    mask2 = cv2.inRange(hsv_car, lower_red2, upper_red2)
                   
                    # Combine both red color masks together
                    combined_red_mask = cv2.bitwise_or(mask1, mask2)
                    red_pixel_count = cv2.countNonZero(combined_red_mask)
                   
                    # For highway traffic, 5% area coverage captures cars far in the distance
                    if (red_pixel_count / total_pixels) > 0.05:
                        red_car_detected_in_frame = True
                       
                        # Draw a vibrant Red bounding box around the targeted car
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
                        cv2.putText(frame, f"Red Car: {conf:.2f}", (x1, y1 - 10),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                    else:
                        # Draw a thin light gray bounding box around other regular vehicles
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (180, 180, 180), 1)

        if red_car_detected_in_frame:
            red_car_frame_count += 1

        # Draw real-time data monitoring HUD
        cv2.rectangle(frame, (10, 10), (340, 90), (0, 0, 0), -1)
        cv2.putText(frame, f"Current Frame: {frame_count}", (20, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, f"Red Car Frames: {red_car_frame_count}", (20, 70),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

        # Handle frame rendering
        cv2.imshow("Red Car Detection Tracker", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
   
    print("\n" + "="*30)
    print(f"Total frames processed: {frame_count}")
    print(f"Frames containing a red car: {red_car_frame_count}")
    print("="*30)

# Execute the updated tracking system script
visualize_red_car_tracking("cars.mp4")

Opening video playback tracker window. Press 'q' to quit.

Total frames processed: 219
Frames containing a red car: 56
